# Create S1 Multiannual Seasonal SBAS Networks

<br>

This notebook demonstrates how to compile Sentinel-1 multiannual, seasonal SBAS networks with [`asf_search`](https://docs.asf.alaska.edu/asf_search/basics/) and order them from ASF's [HyP3-basic](https://hyp3-docs.asf.alaska.edu/hyp3-docs/about/hyp3_basic/) and [HyP3+](https://hyp3-docs.asf.alaska.edu/hyp3-docs/about/hyp3_plus/) on-demand processing serivces. It covers the creation of networks at the full-scene, single-burst, and multi-burst levels. Full-scene interferograms are processed with [GAMMA](https://www.gamma-rs.ch/gamma-software). Single-burst and multi-burst interferograms are processed with [isce2](https://github.com/isce-framework/isce2).

**The notebook creates example SBAS networks. Adjust the paramaters of an SBAS network to order data for your study.**

This workflow demonstrates how to use the `asf_search.SBASNetwork` class to create SBAS networks while excluding seasons likely to cause temporal decorrelation. Because seasonal filtering produces disconnected subnetworks, the `SBASNetwork` class connects them using bridge pairs with temporal baselines of approximately 365 days, or multiples thereof. These bridge pairs create a fully connected network, which is required to perform the least-squares inversion used in SBAS analysis. 

The use of bridge pairs to connect a seasonally filtered network relies on the assumption that where there is strong seasonality, the state of scatterers are likely to be most simmilar when seperated by one year or multiples of one year. The start and end of a season tends to vary year-by-year much more than the middle of a season, so it is often helpful to bridge seasonal SBAS gaps from the center of a targeted season.

You can learn more about the `SBASNetwork` class, and the `Pair` and `Stack` classes that support it in the [`asf_search` example notebooks](https://github.com/asfadmin/Discovery-asf_search/tree/master/examples).

:::{hint} A note regarding full-scene, single-burst, and multi-burst Sentinel-1 data

Sentinel-1 IW scenes are comprised of 3 swaths, each containing ~9 seperate bursts. While individual bursts are geostationary, the scenes that contain them are not. Scenes do not consistently hold the same set of bursts. As a result, a deep stack of full-scene interferograms may have only contain a small area of common spatial coverage extending through the entire stack.

When the goal is consistent data coverage through time, it is more reliable to build stacks out of single-burst or multi-burst interferograms. Using single or multi-bursts also allows you to reduce the volume of downloaded data by working with smaller spatial subsets, tailored to an AOI.

HyP3 on-demand processing supports processing interferograms at the full-scene, single-burst, and multi-burst levels.
:::

:::{hint} This notebook can also be used to generate non-seasonal SBAS networks

If you wish to generate an SBAS network with no seasonal filtering and bridging, omit the following parameters when performing searches below:
- `season`
- `bridge_target_date`
- `bridge_year_threshold`
:::

<hr>

:::{note} **Did you find a bug? Do you have a feature request?**
:::{figure} github_issues.png
:alt: GitHub logo above the word Issues
:width: 10%
:align: left
:target: https://github.com/ASFOpenSARlab/opensarlab_MintPy_Recipe_Book/issues

Explore GitHub Issues on this Jupyter Book's GitHub repository. Find solutions, add to the discussion, or start a new bug report or feature request: <a href="https://github.com/ASFOpenSARlab/opensarlab_MintPy_Recipe_Book/issues" target="_blank" rel="noopener noreferrer">opensarlab_MintPy_Recipe_Book Issues</a>
:::


:::{note} **Have a question related to SAR, ASF data access, or performing SBAS time series analyses with MintPy?**
:::{figure} ASF_support_logo.png
:alt: ASF logo
:width: 10%
:align: left
:target: https://github.com/ASFOpenSARlab/opensarlab_MintPy_Recipe_Book/issues

Contact ASF User Support: <a href="mailto:uso@asf.alaska.edu" target="_blank" rel="noopener noreferrer">uso@asf.alaska.edu</a>
:::

<hr>

## 0. Full-Scene and Single Burst Example

### 0a. Search for a geographic reference scene

Update the `product_id` value below with a product ID for a full Sentinel-1 SLC scene or a single burst SLC representing a geographic reference for the network. 

Use a scene or burst from the start of your time series.

:::{warning} You must use an SLC in the VV polarization

HyP3 InSAR processing does not support VH-polarized data
:::

In [ ]:
import asf_search as asf

geo_reference_id = "S1_257957_IW2_20160112T233838_VV_96AB-BURST" # update this with your reference scene or burst SLC ID (VV only)

results = asf.product_search(geo_reference_id)
reference = results[0]
reference

### 0b. Create an `SBASNetwork`, defining:
- temporal bounds
- seasonal bounds
- geographic reference scene
- perpendicular baseline
- in-season temporal baseline (temporal baseline, excluding those for annual or multiannual seasonal bridge pairs)
- target bridge date (target date from which to bridge seasonal gaps)
- bridge year threshold (number of years across which to create bridge pairs)

:::{hint} Hover over verticies and edges on the plot to see dates and InSAR pair details
:::

In [ ]:
from datetime import datetime, date
import pandas as pd

def get_julian_season(season) -> tuple[int,int]:
    season_start_ts = pd.Timestamp(
        datetime.strptime(f"{season[0]}-0001", "%m-%d-%Y"), tz="UTC"
        )
    season_start_day = season_start_ts.timetuple().tm_yday
    season_end_ts = pd.Timestamp(
        datetime.strptime(f"{season[1]}-0001", "%m-%d-%Y"), tz="UTC"
    )
    season_end_day = season_end_ts.timetuple().tm_yday
    return (season_start_day, season_end_day)

season = ("1-1", "3-1")

sbas = asf.SBASNetwork(
    geo_reference = reference,
    start_date = '2016-01-12', # use the date of your reference scene acquisition
    end_date = '2024-10-02',
    season = get_julian_season(season),
    perpendicular_baseline=400, 
    inseason_temporal_baseline=36,
    bridge_target_date='1-31',
    bridge_year_threshold=1,
    allow_missing_state_vectors=True) # Sometimes, Sentinel-1 metadata is missing state vectors used to determine perpendicular baselines.
                                      # You can opt to remove or include SLCs with missing state vectors.

sbas.plot()

### 0c. Remove and Add Pairs

The `SBASNetwork` class is capable of creating fully connected networks, but it is not guaranteed to. You may occasionally need to alter a network by adding or removing pairs to connect disconnected networks or make adjustments to suit your specific needs.

Add or remove pairs using date strings or lists of date strings.

:::{warning}
The pairs defined below assumes you are using the original geographic reference scene and parameters defined in this notebook.

If you have created your own custom SBAS network, you will update the dates below to dates that exist within your network.
:::

In [ ]:
sbas.add_pairs(("2023-01-11", "2024-01-18"))

sbas.remove_pairs([
    ("2019-01-20", "2019-02-25"),
    ("2019-02-01", "2019-02-25"),
])

sbas.plot()

### 0d. Add a pair including SLCs that fall the original constraints of the SBAS stack you defined

To add pairs that the SBAS Network does not yet know anything about, you must create and pass a new `asf_search.Pair` object.

:::{hint}
You can use this technique to add pairs that extend into filtered seasons or extend beyond the original temporal bounds of the time series
:::

:::{warning}
The pair defined below assumes you are using the original geographic reference scene defined in this notebook.

If you have created your own custom SBAS network, you will update the scene IDs below. 

For burst data, the products must share a burst ID with your geographic reference burst. 

For a full scene, the products must share a path and frame with your geographic reference scene.
:::

In [ ]:
# Search for the primary SLC
primary_id = "S1_257957_IW2_20240211T233925_VV_0137-BURST" 
results = asf.product_search(primary_id)
primary = results[0]

# Search for the secondary SLC
secondary_id = "S1_257957_IW2_20250205T233919_VV_F87E-BURST"
results = asf.product_search(secondary_id)
secondary = results[0]

# Create a Pair object from the primary and secondary SLCs
new_pair = asf.Pair(primary, secondary)

# Add the pair to the SBAS network
sbas.add_pairs(new_pair)

sbas.plot()

### 0e. Prepare to order interferograms in `SBASNetwork` from HyP3

`hyp3_sdk` alternately uses the `submit_insar_isce_burst_job()` and `submit_insar_job()` methods for ordering burst-level and full-scene interferograms.

Since full-scene interferograms are processed with [GAMMA](https://www.gamma-rs.ch/gamma-software), while single-burst interferograms are processed with [isce2](https://github.com/isce-framework/isce2), the `submit_insar_isce_burst_job()` and `submit_insar_job()` methods require different sets of arguments. 

Define the `job_type` for your `SBASNetwork` and the accompanying set of arguments.

In [ ]:
job_type = 'INSAR_ISCE_BURST' # Use 'INSAR_GAMMA' for full-scene interferograms

if job_type == 'INSAR_ISCE_BURST':
    looks = '20x4' # '10x2' or '5x1'
    apply_water_mask = False # optional

elif job_type == 'INSAR_GAMMA':
    looks = '20x4' # '10x2'
    phase_filter_parameter = 0.6, # 0.6 default value
    include_dem = True, # leave set to True, required for mintpy
    include_look_vectors = True # leave set to True, required for mintpy
    apply_water_mask = False # optional

else:
    print(f"job_type must be either INSAR_ISCE_BURST or INSAR_GAMMA, not {job_type}")


### 0f. Order the interferograms in `SBASNetwork` from HyP3 or HyP3+

`sbas.scene_ids` contains the reference and secondary scene IDs for the largest connected network in the `SBASNetwork`. Iterate through them, and order the interferograms from HyP3.

This notebook creates example SBAS networks. Adjust the paramaters of a network to order data for your study.

:::{warning} HyP3 Credit Warning

Executing the following code cell will order interferograms from HyP3-basic or HyP3+ and cost HyP3 credits. 

HyP3-basic credits are free, but based on a monthly quota. If you use all your HyP3-basic credits, you will need to wait for them to be refreshed next month or move to HyP3+ and purchase credits.

HyP3+ is a paid service. Using HyP3+ credits will expend credits that you have paid for.

[More information on HyP3 credits.](https://hyp3-docs.asf.alaska.edu/hyp3-docs/using/credits/)

To process interferograms with HyP3-basic, use the `api_url`: https://hyp3-api.asf.alaska.edu

To process interferograms with HyP3+, use the `api_url`: https://hyp3-plus.asf.alaska.edu

:::

In [ ]:
import hyp3_sdk
from tqdm.auto import tqdm

api_url = 'https://hyp3-api.asf.alaska.edu' # Use https://hyp3-plus.asf.alaska.edu for HyP3+

hyp3 = hyp3_sdk.HyP3(prompt='password', api_url)

project_name = "example_SBASNetwork" # Change this to your desired HyP3 project name

insar_jobs = hyp3_sdk.Batch()

for pair in tqdm(sbas.scene_ids):
    if job_type == 'INSAR_ISCE_BURST':
        insar_jobs += hyp3.submit_insar_isce_burst_job(
            pair.reference, 
            pair.secondary,
            looks=looks,
            apply_water_mask=apply_water_mask,
            name=project_name)

    elif job_type == 'INSAR_GAMMA':
        insar_jobs += hyp3.submit_insar_job(
            pair.reference, 
            pair.secondary,
            looks=looks,
            phase_filter_parameter=phase_filter_parameter,
            include_dem=include_dem,
            include_look_vectors=include_look_vectors,
            apply_water_mask=apply_water_mask,
            name=project_name)
    
print(insar_jobs)

## 1. Multi-Burst Example

The following example walks through the same steps as the section 0 example, but performs the operations on a collection of bursts, which can be ordered from HyP3 as mosaicked multi-burst interferograms. 

### 1a. Create an asf_search.S1MultiBurstGroup object defining a group of Sentinel-1 bursts and subswaths

Refer to [HyP3's mutli-burst guidelines](https://hyp3-docs.asf.alaska.edu/guides/burst_insar_product_guide/#considerations-for-selecting-input-bursts) when assembling a collection of bursts.

S1MultiBurstGroup validation is performed during initialization to prevent the creation of invalid multiburst SBASNetworks that result in failed HyP3 on-demand processing jobs.

In [ ]:
import asf_search as asf

multiburst_group = asf.S1MultiBurstGroup(
    bursts=[
    asf.S1MultiBurst("121_257957", ("IW1", "IW2", "IW3")),
    asf.S1MultiBurst("121_257956", ("IW1", "IW2", "IW3")),
    asf.S1MultiBurst("121_257955", ("IW1", "IW2", "IW3"))
    ]
)
multiburst_group

### 1b. Create a geographic reference S1MultiBurstProduct object

An `S1MultiBurstProduct` contains a collection of Sentinel-1 burst products.

In [ ]:
start_date = '2016-01-12'

reference_multiburst = asf.S1MultiBurstProduct(multiburst_group, start_date)
reference_multiburst

### 1c. Create an SBASNetwork from the S1MultiBurstProduct object

:::{note}

The SBAS plot produced below shows the geographic reference as an `S1MultiBurstProduct` burst ID describing the bursts, swaths, and acquisiton date for the collection of bursts: `121_257957_IW123_121_257956_IW123_121_257955_IW123_20160112`
:::

In [ ]:
from datetime import datetime, date
import pandas as pd

def get_julian_season(season) -> tuple[int,int]:
    season_start_ts = pd.Timestamp(
        datetime.strptime(f"{season[0]}-0001", "%m-%d-%Y"), tz="UTC"
        )
    season_start_day = season_start_ts.timetuple().tm_yday
    season_end_ts = pd.Timestamp(
        datetime.strptime(f"{season[1]}-0001", "%m-%d-%Y"), tz="UTC"
    )
    season_end_day = season_end_ts.timetuple().tm_yday
    return (season_start_day, season_end_day)

season = ("1-1", "3-1")

multiburst_sbas = asf.SBASNetwork.from_geo_reference(
    geo_reference = reference_multiburst,
    start_date = '2016-01-12',
    end_date = '2024-10-02',
    season = get_julian_season(season),
    perpendicular_baseline=400, 
    inseason_temporal_baseline=36,
    bridge_target_date='1-31',
    bridge_year_threshold=1,
    allow_missing_state_vectors=True)

multiburst_sbas.plot()

### 1d. Remove and Add Pairs

The `SBASNetwork` class is capable of creating fully connected networks, but it is not guaranteed to. You may occasionally need to alter a network by adding or removing pairs to connect disconnected networks or make adjustments to suit your specific needs.

Add or remove pairs using date strings or lists of date strings.

In [ ]:
multiburst_sbas.add_pairs([
    ("2023-01-11", "2024-01-18"),
    ("2024-02-11", "2025-02-05"),
    ])

multiburst_sbas.remove_pairs([
    ("2019-01-20", "2019-02-25"),
    ("2019-02-01", "2019-02-25"),
])

multiburst_sbas.plot()

### 1e. Order the multi-burst interferograms in SBASNetwork from HyP3 or HyP3+

`sbas.scene_ids` contains the reference and secondary scene IDs for the largest connected network in the `SBASNetwork`. Iterate through them, and order the interferograms from HyP3.

This notebook creates example SBAS networks. Adjust the paramaters of a network to order data for your study.

:::{warning} HyP3 Credit Warning

Executing the following code cell will order interferograms from HyP3-basic or HyP3+ and cost HyP3 credits. 

HyP3-basic credits are free, but based on a monthly quota. If you use all your HyP3-basic credits, you will need to wait for them to be refreshed next month or move to HyP3+ and purchase credits.

HyP3+ is a paid service. Using HyP3+ credits will expend credits that you have paid for.

[More information on HyP3 credits.](https://hyp3-docs.asf.alaska.edu/hyp3-docs/using/credits/)

To process interferograms with HyP3-basic, use the `api_url`: https://hyp3-api.asf.alaska.edu

To process interferograms with HyP3+, use the `api_url`: https://hyp3-plus.asf.alaska.edu

:::


In [ ]:
import hyp3_sdk
from tqdm.auto import tqdm

api_url = 'https://hyp3-api.asf.alaska.edu' # Use https://hyp3-plus.asf.alaska.edu for HyP3+

hyp3 = hyp3_sdk.HyP3(prompt='password', api_url)

project_name = "example_SBASNetwork_multiburst" # Change this to your desired HyP3 project name

multiburst_jobs = hyp3_sdk.Batch()

for multiburst in tqdm(multiburst_sbas.scene_ids):
    multiburst_jobs += hyp3.submit_insar_isce_multi_burst_job(
        multiburst[0],
        multiburst[1],
        name=project_name,
        apply_water_mask=False,
        looks='20x4', # '20x4', '10x2', '5x1'
    )

## 2. Check the status of the jobs you ordered on [Vertex](https://search.asf.alaska.edu/)

Go to [Vertex](https://search.asf.alaska.edu/) and check the status of your jobs with an `On-Demand` search.

:::{figure} on_demand_vertex.png 
:alt: Image of a Vertex On-Demand search showing pending jobs

:::

<hr>

*Author: Alex Lewandowski; Alaska Satellite Facility*